# Setting and Loading VWB Environment Variables

This notebook ensures that key workspace-level variables — like the Google Cloud project, workspace buckets, and CDR — are available to every notebook in this Jupyter environment.

It works by:
1. Writing the variables to your `~/.bashrc` file (so they persist across sessions).
2. Using a helper function to load them into Python’s `os.environ`.
3. Allowing other notebooks to simply import these variables instead of redefining them.

---

### Variables that will be set
- `GOOGLE_CLOUD_PROJECT`
- `WORKSPACE_BUCKET`
- `WORKSPACE_CDR`


We will use the WB CLI commands to dynamically obtain our environment variables and set them. This is the preferred method as updates to the datasets with each release can occur and overwrite pre-existing data.

## Step 1: Read and write environment variables to `.bashrc`

This step:
- extract main env variables from wb resource
- Checks whether `~/.bashrc` exists (creates it if not).
- Appends export statements for the four variables.
- If a variable already exists, it replaces its old value with the new one.

You’ll only need to run this cell once unless you want to change the variable values.

In [ ]:
import json
import os
import subprocess

# Fixed CDR dataset
TARGET_CDR_DATASET = "C2024Q3R9"

# --- Get workspace info ---
workspace = json.loads(
    subprocess.run(
        ["wb", "workspace", "describe", "--format=json"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
)

os.environ["GOOGLE_CLOUD_PROJECT"] = workspace.get("googleProjectId", "")

# --- Get resources ---
resources = json.loads(
    subprocess.run(
        ["wb", "resource", "list", "--format=json"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
)

# Initialize variables
os.environ["WORKSPACE_CDR"] = ""
os.environ["WORKSPACE_BUCKET"] = ""

# --- Process resources ---
for resource in resources:
    resource_type = resource.get("resourceType", "")

    # --- BUCKETS ---
    if resource_type == "GCS_BUCKET":
        resource_id = resource.get("id", "")
        bucket_name = resource.get("bucketName", "")

        print(
            f"Found bucket: id={resource_id}, "
            f"bucketName={bucket_name}"
        )

        if "polypharmacy-bucket" in resource_id and bucket_name:
            os.environ["WORKSPACE_BUCKET"] = f"gs://{bucket_name}"

    # --- DATASETS ---
    elif resource_type in {"BQ_DATASET", "BIGQUERY_DATASET"}:
        dataset_id = resource.get("datasetId", "")

        if dataset_id == TARGET_CDR_DATASET:
            project_id = (
                resource.get("projectId")
                or os.environ["GOOGLE_CLOUD_PROJECT"]
            )

            if project_id:
                os.environ["WORKSPACE_CDR"] = (
                    f"{project_id}.{TARGET_CDR_DATASET}"
                )
            else:
                os.environ["WORKSPACE_CDR"] = TARGET_CDR_DATASET

# --- Validate CDR dataset ---
if os.environ["WORKSPACE_CDR"]:
    print(
        f"✅ WORKSPACE_CDR set to: "
        f"{os.environ['WORKSPACE_CDR']}"
    )
else:
    print(
        f"❌ Dataset {TARGET_CDR_DATASET} was not found "
        "in the workspace resources"
    )

# --- Validate bucket ---
if not os.environ["WORKSPACE_BUCKET"]:
    print("⚠️ WORKSPACE_BUCKET not found")

# --- Print variables ---
print("\nVariables extracted:")
for variable in [
    "GOOGLE_CLOUD_PROJECT",
    "WORKSPACE_BUCKET",
    "WORKSPACE_CDR",
]:
    print(
        f"{variable}: "
        f"{os.environ.get(variable) or 'NOT FOUND'}"
    )

# --- Save to .bashrc, replacing previous entries ---
bashrc_path = os.path.expanduser("~/.bashrc")

existing_lines = []
if os.path.exists(bashrc_path):
    with open(bashrc_path, "r", encoding="utf-8") as file:
        existing_lines = file.readlines()

variables_to_write = [
    "GOOGLE_CLOUD_PROJECT",
    "WORKSPACE_BUCKET",
    "WORKSPACE_CDR",
]

# Remove previous definitions
filtered_lines = [
    line
    for line in existing_lines
    if not any(
        line.startswith(f"export {variable}=")
        for variable in variables_to_write
    )
]

# Append current values
with open(bashrc_path, "w", encoding="utf-8") as file:
    file.writelines(filtered_lines)

    if filtered_lines and not filtered_lines[-1].endswith("\n"):
        file.write("\n")

    file.write("\n# Verily Workbench variables\n")

    for variable in variables_to_write:
        value = os.environ.get(variable)
        if value:
            file.write(f'export {variable}="{value}"\n')

print(f"\n✅ Saved environment variables to {bashrc_path}")
print("Run `source ~/.bashrc` to load them in the current shell.")

## Step 2: Load Environment Variables in Python and test

The following function reads your `~/.bashrc` file and sets each variable in Python’s runtime environment (`os.environ`), so you can use them directly in your code.

Any other notebook can reuse this function to reload environment variables at the start.

In [ ]:
import os

# In your notebook, just run this:
with open(os.path.expanduser("~/.bashrc"), 'r') as f:
    for line in f:
        if line.strip().startswith('export '):
            parts = line.strip().replace('export ', '').split('=', 1)
            if len(parts) == 2:
                var_name = parts[0].strip()
                var_value = parts[1].strip().strip("'\"")
                
                # SKIP PATH completely!
                if var_name == 'PATH':
                    continue  # Skip this line
                    
                os.environ[var_name] = var_value

# Now use them
print(f"GOOGLE_CLOUD_PROJECT = {os.environ.get('GOOGLE_CLOUD_PROJECT')}")
print(f"WORKSPACE_BUCKET = {os.environ.get('WORKSPACE_BUCKET')}")

**Confirm that the environment variables are correctly set.**

In [ ]:
!echo $WORKSPACE_CDR

In [ ]:
!echo $WORKSPACE_TEMP_BUCKET